# Enrichment analysis: fetching KEGG IDs for MS2Query results
- match InChIKeys to PubChem CIDs
- match PubChem CIDs to KEGG IDs
to annotate features identified by MS2Query with KEGG pathways.

In [ ]:
from collections import defaultdict

import pandas as pd
import pubchempy as pcp
from acore.io.kegg import lookup_cid_to_kegg_id


def parse_compound_pathway_mapping(raw_mapping: str) -> dict[str, list[str]]:
    """Parse tab-delimited KEGG-style compound/pathway mappings into a dictionary."""
    compound_to_pathways: defaultdict[str, list[str]] = defaultdict(list)

    for line in raw_mapping.strip().strip("'").splitlines():
        compound_id, pathway_id = line.split("\t", maxsplit=1)
        compound_to_pathways[compound_id].append(pathway_id)

    return dict(compound_to_pathways)


compounds = {}

In [ ]:
fname_ms2query = "results_prepared/output_ms2query_Linked_data.tsv"

In [ ]:
ms2query_results = pd.read_csv(fname_ms2query, index_col=0, sep="\t").drop_duplicates(
    subset=["inchikey", "smiles"]
)
ms2query_results.head()

In [ ]:
to_lookup = ms2query_results[
    [
        "ms2query_model_prediction",
        "precursor_mz_difference",
        "precursor_mz_query_spectrum",
        "precursor_mz_analog",
        "inchikey",
        "analog_compound_name",
        "smiles",
    ]
].drop_duplicates(subset=["inchikey", "smiles"])
to_lookup

Find PubChem CIDs for the InChIKeys

In [ ]:
for _inchikey in to_lookup.inchikey.unique():
    if _inchikey in compounds:
        continue
    print(f"Looking up {_inchikey} in PubChem...")
    compounds[_inchikey] = pcp.get_compounds(_inchikey, namespace="inchikey")
compounds

Then look up KEGG IDs for those CIDs

In [ ]:
cids = [c.cid for pcp_list in compounds.values() for c in pcp_list if c.cid is not None]
kegg_compounds = lookup_cid_to_kegg_id(cids)
kegg_compounds

In [ ]:
inchikey_to_kegg = []
for inchikey, pcp_list in compounds.items():
    for c in pcp_list:
        if c.cid is not None and c.cid in kegg_compounds:
            inchikey_to_kegg.append((inchikey, kegg_compounds[c.cid]))
inchikey_to_kegg

In [ ]:
inchikey_to_kegg = pd.DataFrame(inchikey_to_kegg, columns=["inchikey", "kegg_id"])
inchikey_to_kegg

And finally find KEGG pathways for those KEGG IDs.

In [ ]:
inchikey_to_kegg = inchikey_to_kegg.join(
    to_lookup.reset_index().set_index("inchikey"), on="inchikey"
)
inchikey_to_kegg

In [ ]:
fname = "results_prepared/inchikey_to_kegg.csv"
inchikey_to_kegg.to_csv(fname, index=False)

Done.